In [15]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from IPython.display import SVG, display
from holo.pointers import Pointer
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY, OUR_DATASET_DIRECTORY_2, OUR_DATASET_DIRECTORY_3
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, GenerationStats
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

sys.version_info(major=3, minor=13, micro=12, releaselevel='final', serial=0)


In [16]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [26]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [27]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model.load("model_1.6M", versionID=6, device=torch.device("cuda:0"))
model.show_infos()

loading the tokenizer from: /mnt/c/Users/jaduc/OneDrive/Desktop/univeristé/M2_info/PFE/PFE_LLM_art_generation/models_save/model_1.6M/tokenizer.json
loading the historique from: /mnt/c/Users/jaduc/OneDrive/Desktop/univeristé/M2_info/PFE/PFE_LLM_art_generation/models_save/model_1.6M/versions/version_0006_trained-5epoches/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=128, window_pattern='SSSL')
1_589_356 total params (embeding: 327_680 | last layer: 81_920 | transformer: 1_179_744)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [28]:
print(f"trained for {model.nb_epoches_done} epochs")


trained for 5 epochs


In [30]:
try:
    if "dataset" in globals() : raise FileExistsError
    # TO DO : ajouter les param du gcode ici
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY, context_size=model.context_size,
        tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode, use_gcode = True, use_relative_gcode=True)
except FileExistsError: pass

In [31]:
N = 13
print(f"using: {dataset.samples[N].svg_file}")
#start = dataset.samples[N].txt[: model.context_size]
#print(start); break
first_chunck = next(c for c in dataset.chunks if c.indexes.svgIndex==N)
start = first_chunck.text
statsPtr: Pointer[GenerationStats] = Pointer()
results = []
for txt in model.generate_flow(
        start=start, decode_batch=256, temperature=1.0, top_k=5, 
        max_tokens=320000, max_time=2*60, statsPtr=statsPtr):
    results.append(txt)
    print(txt, end="", flush=True)

using: /mnt/c/Users/jaduc/OneDrive/Desktop/univeristé/M2_info/PFE/PFE_LLM_art_generation/dataset/samples_100/0013_circle_packing.svg
808.214.365.84342.648.66644.225.665.251.222.648.478.174.846.252.666.256.808.605.258.479.195.192.255.819.196.256.182.809.697.252.638.6064"/><line y2="266.256" style="fill:none;" x1="266.6198"/><line y2="275.8026" style="fill:none;" x1="3.256" style="fill:none;" x1="277.639" y1="271.817"/><line y2="269.7939" style="fill:none;" x1="272.269" style="fill:none;" x1="275.808" y1="2="145.809" y1="146.8073"/><line y2="121.258" style="fill:none;" x1="286.614" y1="307.79"/><line y2="120.619" style="fill:none;" x1="307.799" x2="319.804" y1="126.736"/><line y2="116.6079" style="fill:none;" x1="319.804" y1="331.81"/><line y2="137.8098" style="fill:none;" x1="376.7921" y1="346.666"/><line y2="148.2173" style="fill:none;" x1="130.606" y1="152.61"/><line y2="309.6024" style="fill:none;" x1="140.696" y1="147.7964"/><line y2="215.694" style="fill:none;" x1="33.7979" x2="190

In [32]:
print(start)

<|output_start|>G20
G17
G90
G00 X0.0000 Y9.3750
G01 X9.3750 Y9.3750
G01 X9.3750 Y0.0000
G01 X0.0000 Y0.0000
G01 X0.0000 Y9.3750
G00 X5.9714 Y8.0723
G01 X5.9714 Y8.0683
G01 X5.9713 Y8.0644
G01 X5.9711 Y8.0604
G01 X5.9708 Y8.0565
G01 X5.9704 Y8.0526
G01 X5.9700 Y8.0486
G01 X5.9695 Y8.0447
G01 X5.9689 Y8.0408
G01 X5.9683 Y8.0369
G01 X5.9675 Y8.0330
G01 X5.9667 Y8.0292
G01 X5.9659 Y8.0253
G01 X5.9649 Y8.0215
G01 X5.9639 Y8.0177
G01 X5.9628 Y8.0139
G01 X5.9616 Y8.0102
G01 X5.9603 Y8.0064
G01 X5.9590 Y8.0027
G01 X5.9576 Y7.9990
G01 X5.9561 Y7.9954
G01 X5.9546 Y7.9917
G01 X5.9529 Y7.9881
G01 X5.9513 Y7.9846
G01 X5.9495 Y7.9810
G01 X5.9477 Y7.9775
G01 X5.9458 Y7.9741
G01 X5.9438 Y7.9706
G01 X5.9418 Y7.9673
G01 X5.9397 Y7.9639
G01 X5.9375 Y7.9606
G01 X5.9353 Y7.9573
G01 X5.9330 Y7.9541
G01 X5.9307 Y7.9510
G01 X5.9283 Y7.9478
G01 X5.9258 Y7.9448
G01 X5.9233 Y7.9417
G01 X5.9207 Y7.9388
G01 X5.9180 Y7.9358
G01 X5.9153 Y7.9330
G01 X5.9125 Y7.9302
G01 X5.9097 Y7.9274
G01 X5.9069 Y7.9247
G01 X5.9039 

In [33]:
results.insert(0, start)
print(statsPtr.value)
print(f" -> {statsPtr.value.nb_tokens / statsPtr.value.gen_time:.2f} tokens/sec")

GenerationStats(nb_tokens=2847, gen_time=120.00645167499988, stop_reason='reached max_time')
 -> 23.72 tokens/sec


In [34]:
print(results)


['<|output_start|>G20\nG17\nG90\nG00 X0.0000 Y9.3750\nG01 X9.3750 Y9.3750\nG01 X9.3750 Y0.0000\nG01 X0.0000 Y0.0000\nG01 X0.0000 Y9.3750\nG00 X5.9714 Y8.0723\nG01 X5.9714 Y8.0683\nG01 X5.9713 Y8.0644\nG01 X5.9711 Y8.0604\nG01 X5.9708 Y8.0565\nG01 X5.9704 Y8.0526\nG01 X5.9700 Y8.0486\nG01 X5.9695 Y8.0447\nG01 X5.9689 Y8.0408\nG01 X5.9683 Y8.0369\nG01 X5.9675 Y8.0330\nG01 X5.9667 Y8.0292\nG01 X5.9659 Y8.0253\nG01 X5.9649 Y8.0215\nG01 X5.9639 Y8.0177\nG01 X5.9628 Y8.0139\nG01 X5.9616 Y8.0102\nG01 X5.9603 Y8.0064\nG01 X5.9590 Y8.0027\nG01 X5.9576 Y7.9990\nG01 X5.9561 Y7.9954\nG01 X5.9546 Y7.9917\nG01 X5.9529 Y7.9881\nG01 X5.9513 Y7.9846\nG01 X5.9495 Y7.9810\nG01 X5.9477 Y7.9775\nG01 X5.9458 Y7.9741\nG01 X5.9438 Y7.9706\nG01 X5.9418 Y7.9673\nG01 X5.9397 Y7.9639\nG01 X5.9375 Y7.9606\nG01 X5.9353 Y7.9573\nG01 X5.9330 Y7.9541\nG01 X5.9307 Y7.9510\nG01 X5.9283 Y7.9478\nG01 X5.9258 Y7.9448\nG01 X5.9233 Y7.9417\nG01 X5.9207 Y7.9388\nG01 X5.9180 Y7.9358\nG01 X5.9153 Y7.9330\nG01 X5.9125 Y7.9302\nG